# Phi multitask fine-tuning

Historical experiment source: `bench_gemma/FewZeroShot/LLM_Phi4_mini_Fine_tune_Multitask.ipynb`. Outputs and stale result commentary were removed for publication. Scientific logic is retained; these experiments and their reported results have not been rerun or validated here.

This historical notebook retains the first-token scorer involved in the Phi Heart 0.5 issue. See the [corrected scorer](../scoring.py); new evaluation is required before replacing historical results.

In [ ]:
import os

DATA_ROOT = os.environ.get("PHI_DATA_ROOT", os.getcwd())
HF_CACHE  = f"{DATA_ROOT}/hf_cache"
OUTPUT_DIR    = f"{DATA_ROOT}/phi4_multitask"
RESULTS_FILE  = f"{DATA_ROOT}/phi4_multitask_results.json"

import os
os.environ["HF_HOME"]            = HF_CACHE
os.environ["HF_HUB_CACHE"]       = HF_CACHE
os.environ["TRANSFORMERS_CACHE"] = HF_CACHE
os.environ["HF_DATASETS_CACHE"]  = HF_CACHE
os.makedirs(HF_CACHE, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

import sys, gc, json, math, time, random, logging
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from transformers import (
    AutoModelForCausalLM, AutoTokenizer, AutoConfig,
    TrainingArguments, Trainer,
    DataCollatorForLanguageModeling,
)
from datasets import Dataset
from peft import LoraConfig, get_peft_model

from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
)
from tqdm.auto import tqdm

import transformers, peft, datasets, accelerate
assert torch.cuda.is_available(), "GPU не виден"
print(f"torch {torch.__version__} | cuda {torch.cuda.is_available()} | {torch.cuda.get_device_name(0)}")
print(f"transformers {transformers.__version__} | peft {peft.__version__} | "
      f"datasets {datasets.__version__} | accelerate {accelerate.__version__}")
print(f"HF cache: {HF_CACHE}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

LOG_FILE = f"{DATA_ROOT}/phi4_multitask.log"
logger = logging.getLogger("experiment")
logger.setLevel(logging.INFO)
logger.handlers.clear()
fh = logging.FileHandler(LOG_FILE, mode="w", encoding="utf-8")
fh.setFormatter(logging.Formatter("%(asctime)s | %(message)s", datefmt="%Y-%m-%d %H:%M:%S"))
logger.addHandler(fh)
ch = logging.StreamHandler(sys.stdout)
ch.setFormatter(logging.Formatter("%(message)s"))
logger.addHandler(ch)
_orig_print = print
def print(*args, **kwargs):
    logger.info(" ".join(str(a) for a in args))
_orig_print(f"Лог: {LOG_FILE}")

In [ ]:
import os
from huggingface_hub import login
if os.environ.get("HF_TOKEN"):
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

In [ ]:
model_name = "microsoft/Phi-4-mini-instruct"

DATASETS_TO_USE = ["bank", "blood", "california", "credit_g",
                   "diabetes", "heart", "income", "car", "jungle"]

PER_DATASET_CAP = 15000
NUM_EPOCHS = 3
BATCH_SIZE = 16
GRAD_ACCUM = 2
MAX_SEQ_LEN = 512
LEARNING_RATE = 2e-4
WARMUP_STEPS = 100
WEIGHT_DECAY = 0.01
BATCH_SIZE_EVAL = 32
SEED = 42

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# 1. Загрузка данных

In [ ]:
DATA_DIR = "datasets"

BANK_FEATURE_MAPPINGS = {
    "V1": "Age", "V2": "Job", "V3": "Martial", "V4": "Education",
    "V5": "Default", "V6": "Balance", "V7": "Housing", "V8": "Loan",
    "V9": "Contact", "V10": "Day of Week", "V11": "Month", "V12": "Duration",
    "V13": "Campaign", "V14": "Pdays", "V15": "Previous", "V16": "Poutcome",
}
BLOOD_FEATURE_MAPPINGS = {
    "V1": "Recency", "V2": "Frequency", "V3": "Monetary", "V4": "Time",
}

def _binary(pos, neg): return [neg, pos]

DATASET_FILES = {
    "bank":       ("bank_marketing_openml_1461.parquet",      "Class",         BANK_FEATURE_MAPPINGS,  "y",             "2",     None),
    "blood":      ("blood_openml_1464.parquet",               "Class",         BLOOD_FEATURE_MAPPINGS, "Donated blood", "2",     None),
    "california": ("california_housing_openml_44090.parquet", "price",         None,                   None,            "True",  None),
    "credit_g":   ("credit_g_openml_31.parquet",              "class",         None,                   None,            "good",  None),
    "diabetes":   ("diabetes_pima.parquet",                   "Outcome",       None,                   None,            None,    None),
    "heart":      ("heart_failure.parquet",                   "HeartDisease",  None,                   None,            None,    None),
    "income":     ("income_openml_1590.parquet",              "class",         None,                   None,            ">50K",  None),
    "car":        ("car_openml_40975.parquet",                "class",         None,                   None,            None,    ["unacc", "acc", "good", "vgood"]),
    "jungle":     ("jungle_openml_41027.parquet",             "class",         None,                   None,            None,    ["b", "d", "w"]),
}

DATASETS = {
    "bank": dict(prompt_config=dict(
        task="Predict whether a bank client will subscribe",
        labels=_binary("yes", "no"),
        entity="subscription",
        question="Will this client subscribe?")),
    "blood": dict(prompt_config=dict(
        task="Predict whether a person donated blood",
        labels=_binary("yes", "no"),
        entity="Donor",
        question="Did this person donate blood?")),
    "california": dict(prompt_config=dict(
        task="Predict whether house price is above median",
        labels=_binary("yes", "no"),
        entity="House",
        question="Is this house price above median?")),
    "credit_g": dict(prompt_config=dict(
        task="Classify credit risk as good or bad",
        labels=_binary("good", "bad"),
        entity="Client",
        question="Is this client a good credit risk?")),
    "diabetes": dict(prompt_config=dict(
        task="Predict whether a patient has diabetes",
        labels=["no", "yes"],
        entity="Patient",
        question="Does this patient have diabetes?")),
    "heart": dict(prompt_config=dict(
        task="Predict whether a patient has heart disease",
        labels=["0", "1"],
        entity="Patient",
        question="Does this patient have heart disease based on clinical features?")),
    "income": dict(prompt_config=dict(
        task="Predict whether a person's annual income exceeds $50,000",
        labels=_binary(">50K", "<=50K"),
        entity="Person",
        question="Does this person earn more than 50K a year based on census data?")),
    "car": dict(prompt_config=dict(
        task="Predict car evaluation (unacceptable, acceptable, good, very good)",
        labels=["unacceptable", "acceptable", "good", "very good"],
        entity="Car",
        question="What is the evaluation of this car?")),
    "jungle": dict(prompt_config=dict(
        task="Predict the endgame result of Jungle Chess (Dou Shou Qi)",
        labels=["black_win", "draw", "white_win"],
        entity="Game Position",
        question="Based on the rank, file, and strength of the white and black pieces, what is the game result? (White wins, Black wins, or Draw)")),
}


def encode_target(df, target_name, num_classes, pos_hint=None, multiclass_order=None):
    if pd.api.types.is_integer_dtype(df[target_name]):
        df[target_name] = df[target_name].astype(int)
        return df
    raw = df[target_name].unique().tolist()
    raw_strs = [str(v) for v in raw]
    if num_classes == 2:
        if pos_hint is not None and pos_hint in raw_strs:
            pos_val = raw[raw_strs.index(pos_hint)]
        else:
            pos_val = sorted(raw, key=str)[-1]
        df[target_name] = df[target_name].map({v: (1 if v == pos_val else 0) for v in raw}).astype(int)
    else:
        if multiclass_order is not None:
            mapping = {raw[raw_strs.index(h)]: i for i, h in enumerate(multiclass_order)}
        else:
            mapping = {v: i for i, v in enumerate(sorted(raw, key=str))}
        df[target_name] = df[target_name].map(mapping).astype(int)
    return df


def load_one(name):
    fname, target_name, rename_feats, rename_target, pos_hint, order = DATASET_FILES[name]
    df = pd.read_parquet(f"{DATA_DIR}/{fname}")
    if rename_feats:
        df = df.rename(columns=rename_feats)
    if rename_target and target_name in df.columns:
        df = df.rename(columns={target_name: rename_target})
        target_name = rename_target
    num_cls = len(DATASETS[name]["prompt_config"]["labels"])
    df = encode_target(df, target_name, num_cls, pos_hint=pos_hint, multiclass_order=order)
    feature_names = [c for c in df.columns if c != target_name]
    return df, feature_names, target_name


def split_dataset(df, target_name, test_size=0.2, val_size=0.25, seed=SEED):
    train_val, test = train_test_split(df, test_size=test_size, random_state=seed, stratify=df[target_name])
    train, val = train_test_split(train_val, test_size=val_size, random_state=seed, stratify=train_val[target_name])
    return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)


loaded = {}
for name in DATASETS_TO_USE:
    print(f"Загружаю {name}...")
    df, feature_names, target_name = load_one(name)
    train_df, val_df, test_df = split_dataset(df, target_name)
    print(f"  train={len(train_df)} val={len(val_df)} test={len(test_df)}; "
          f"features={len(feature_names)}; classes={sorted(df[target_name].unique().tolist())}")
    loaded[name] = dict(
        prompt_config=DATASETS[name]["prompt_config"],
        feature_names=feature_names,
        target_name=target_name,
        train_df=train_df, val_df=val_df, test_df=test_df,
    )

In [ ]:
for name, info in loaded.items():
    counts = info["train_df"][info["target_name"]].value_counts().sort_index()
    labels = info["prompt_config"]["labels"]
    parts = [f"{labels[int(k)]}: {v}" for k, v in counts.items()]
    print(f"{name:12s} train распределение: " + ", ".join(parts))

# 2. Вспомогательные функции

In [ ]:
def row_to_text_template(row, feature_names):
    parts = []
    for feature in feature_names:
        value = row[feature]
        if isinstance(value, (int, np.integer)):
            parts.append(f"The value of {feature} is {value}.")
        elif isinstance(value, (float, np.floating)):
            parts.append(f"The value of {feature} is {value:.2f}.")
        else:
            parts.append(f"The category of {feature} is {value}.")
    return " ".join(parts)

def build_system_prompt(prompt_config):
    labels_str = "', '".join(prompt_config["labels"])
    return (f"You are a classifier. {prompt_config['task']}. "
            f"Answer with only one word from: '{labels_str}'.")

def build_messages(row, feature_names, prompt_config, target_name=None, include_target=False):
    input_text = row_to_text_template(row, feature_names)
    messages = [
        {"role": "system", "content": build_system_prompt(prompt_config)},
        {"role": "user",
         "content": f"{prompt_config['entity']} information: {input_text}\n{prompt_config['question']}"},
    ]
    if include_target:
        messages.append({"role": "assistant",
                         "content": prompt_config["labels"][int(row[target_name])]})
    return messages

In [ ]:
def parse_prediction(response, prompt_config):
    response = response.lower().strip().rstrip(".,!? ")
    labels = [l.lower() for l in prompt_config["labels"]]
    for i, lab in enumerate(labels):
        if response == lab or response.startswith(lab):
            return i
    words = response.split()
    for i, lab in enumerate(labels):
        if lab in words:
            return i
    return 0

def compute_metrics(y_true, y_pred, y_prob, num_classes):
    acc = accuracy_score(y_true, y_pred)
    if num_classes == 2:
        f1 = f1_score(y_true, y_pred, zero_division=0)
        pr = precision_score(y_true, y_pred, zero_division=0)
        rc = recall_score(y_true, y_pred, zero_division=0)
        try:
            roc = roc_auc_score(y_true, y_prob[:, 1] if y_prob.ndim == 2 else y_prob)
        except ValueError:
            roc = float("nan")
    else:
        f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
        pr = precision_score(y_true, y_pred, average="macro", zero_division=0)
        rc = recall_score(y_true, y_pred, average="macro", zero_division=0)
        try:
            roc = roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")
        except ValueError:
            roc = float("nan")
    return {"ROC-AUC": roc, "F1": f1, "Accuracy": acc, "Precision": pr, "Recall": rc}

def bootstrap_metrics(y_true, y_pred, y_prob, num_classes, n_iter=1000):
    scores = []
    for i in range(n_iter):
        idx = resample(np.arange(len(y_true)), random_state=i + 1)
        try:
            m = compute_metrics(y_true[idx], y_pred[idx], y_prob[idx], num_classes)
            scores.append([m["ROC-AUC"], m["F1"], m["Accuracy"], m["Precision"], m["Recall"]])
        except ValueError:
            continue
    arr = np.asarray(scores)
    means = np.nanmean(arr, axis=0)
    stds = np.nanstd(arr, axis=0, ddof=1)
    names = ["ROC-AUC", "F1", "Accuracy", "Precision", "Recall"]
    return {n: f"{m:.4f}\u00b1{s:.4f}" for n, m, s in zip(names, means, stds)}

def flush_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# 2.1 Загрузка модели Phi-4-mini-instruct

In [ ]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}; total memory = "
          f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

tokenizer = AutoTokenizer.from_pretrained(
    model_name, cache_dir=HF_CACHE, use_fast=True, trust_remote_code=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

# 3. Fine-tuning с LoRA

## Подготовка данных для Fine-tuning

Для каждого датасета балансируем классы (upsampling минорных), отрезаем до `PER_DATASET_CAP` строк, превращаем в chat-template и склеиваем в единый train. Перемешиваем.

In [ ]:
def balance_dataset(train_df, target_name, seed=SEED):
    counts = train_df[target_name].value_counts()
    n_max = counts.max()
    parts = []
    for cls in counts.index:
        sub = train_df[train_df[target_name] == cls]
        if len(sub) < n_max:
            sub = resample(sub, replace=True, n_samples=n_max, random_state=seed + int(cls))
        parts.append(sub)
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)

for name, info in loaded.items():
    bal = balance_dataset(info["train_df"], info["target_name"])
    info["train_balanced"] = bal
    counts = bal[info["target_name"]].value_counts().sort_index().to_dict()
    print(f"{name:12s} balanced train = {len(bal):>6d}  classes: {counts}")

In [ ]:
def build_training_texts(df, feature_names, target_name, prompt_config, tokenizer, cap):
    if cap is not None and len(df) > cap:
        df = df.sample(n=cap, random_state=SEED).reset_index(drop=True)
    texts = []
    for _, row in df.iterrows():
        messages = build_messages(row, feature_names, prompt_config,
                                  target_name=target_name, include_target=True)
        texts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False))
    return texts

all_texts = []
per_dataset_counts = {}
t0 = time.time()
for name in DATASETS_TO_USE:
    info = loaded[name]
    texts = build_training_texts(
        info["train_balanced"], info["feature_names"], info["target_name"],
        info["prompt_config"], tokenizer, cap=PER_DATASET_CAP,
    )
    per_dataset_counts[name] = len(texts)
    all_texts.extend(texts)
    print(f"  {name:12s} → {len(texts)} примеров")

print(f"\nИтого текстов: {len(all_texts)} (собрали за {time.time()-t0:.1f} с)")

rng = random.Random(SEED)
rng.shuffle(all_texts)
train_dataset = Dataset.from_dict({"text": all_texts})

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=MAX_SEQ_LEN, padding=False)

tokenized_dataset = train_dataset.map(
    tokenize_function, batched=True,
    remove_columns=train_dataset.column_names,
    num_proc=4, desc="Tokenizing",
)
print(f"Токенизировано: {len(tokenized_dataset)}")
lengths = [len(x) for x in tokenized_dataset["input_ids"][:5000]]
print(f"len p50/p95/p99 = {int(np.percentile(lengths,50))}/{int(np.percentile(lengths,95))}/{int(np.percentile(lengths,99))}")

## Настройка LoRA и обучение

In [ ]:
# Phi-4 использует custom modeling code (configuration_phi3.py / modeling_phi3.py),
# поэтому подгружаем сначала AutoConfig с trust_remote_code=True
config = AutoConfig.from_pretrained(model_name, cache_dir=HF_CACHE, trust_remote_code=True)

model_lora = AutoModelForCausalLM.from_pretrained(
    model_name,
    config=config,
    cache_dir=HF_CACHE,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="sdpa",
    trust_remote_code=True,
)

model_lora.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)

# Phi-4 имеет fused-проекции:
#   attention   : qkv_proj (Q+K+V), o_proj
#   MLP (SwiGLU): gate_up_proj (gate+up), down_proj
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["qkv_proj", "o_proj", "gate_up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model_lora = get_peft_model(model_lora, lora_config)
model_lora.print_trainable_parameters()

training_args_lora = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    bf16=True,
    tf32=True,
    logging_steps=20,
    logging_first_step=True,
    save_strategy="epoch",
    save_total_limit=1,
    optim="adamw_torch_fused",
    warmup_steps=WARMUP_STEPS,
    max_grad_norm=1.0,
    weight_decay=WEIGHT_DECAY,
    report_to="none",
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    group_by_length=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    torch_compile=False,
    seed=SEED,
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

if torch.cuda.is_available():
    print(f"\nGPU allocated memory after setup: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
n_examples = len(tokenized_dataset)
steps_per_epoch = math.ceil(n_examples / (BATCH_SIZE * GRAD_ACCUM))
print(f"\nНачинаем обучение на {NUM_EPOCHS} эпох")
print(f"Batch size: {BATCH_SIZE}, Grad accum: {GRAD_ACCUM}, Effective: {BATCH_SIZE * GRAD_ACCUM}")
print(f"Примеров: {n_examples}; шагов/эпоху: ~{steps_per_epoch}; всего шагов: ~{steps_per_epoch * NUM_EPOCHS}")

trainer = Trainer(
    model=model_lora,
    args=training_args_lora,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

start_time = time.time()
trainer.train()
train_time = time.time() - start_time

print(f"\nОбучение завершено за {train_time:.1f}s ({train_time/60:.1f} мин, {train_time/3600:.2f} ч)")
if torch.cuda.is_available():
    print(f"GPU peak memory: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

del trainer
flush_gpu()
model_lora.eval()

# 4. Оценка на test по каждому датасету

In [ ]:
def predict_batch(prompts, prompt_config, model, tokenizer, device, max_new_tokens=3):
    inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True,
                       max_length=MAX_SEQ_LEN).to(device)
    with torch.no_grad():
        out = model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            output_scores=True,
            return_dict_in_generate=True,
        )
    input_len = inputs.input_ids.shape[1]
    generated = out.sequences[:, input_len:]
    responses = [r.strip().lower()
                 for r in tokenizer.batch_decode(generated, skip_special_tokens=True)]

    first_logits = out.scores[0]
    label_ids = []
    for lab in prompt_config["labels"]:
        # пробуем с пробелом впереди — у BPE-токенизаторов (как у Phi-4) почти всегда так
        for variant in (f" {lab}", lab):
            ids = tokenizer.encode(variant, add_special_tokens=False)
            if ids:
                label_ids.append(ids[0])
                break
    selected = torch.stack([first_logits[:, lid] for lid in label_ids], dim=1)
    probs = F.softmax(selected, dim=1).detach().cpu().numpy()

    del inputs, out, first_logits, selected, generated
    flush_gpu()
    return responses, probs

def evaluate_dataset(name, info, model, tokenizer, device):
    test_df = info["test_df"]
    feature_names = info["feature_names"]
    target_name = info["target_name"]
    prompt_config = info["prompt_config"]
    num_classes = len(prompt_config["labels"])

    y_true, y_pred, y_prob = [], [], []
    n_batches = math.ceil(len(test_df) / BATCH_SIZE_EVAL)
    t0 = time.time()
    for start in tqdm(range(0, len(test_df), BATCH_SIZE_EVAL),
                      total=n_batches, desc=f"eval {name}", leave=False):
        batch_df = test_df.iloc[start:start + BATCH_SIZE_EVAL]
        prompts = [
            tokenizer.apply_chat_template(
                build_messages(row, feature_names, prompt_config, include_target=False),
                tokenize=False, add_generation_prompt=True,
            )
            for _, row in batch_df.iterrows()
        ]
        responses, probs = predict_batch(prompts, prompt_config, model, tokenizer, device)
        for (_, row), resp, p in zip(batch_df.iterrows(), responses, probs):
            y_true.append(int(row[target_name]))
            y_pred.append(parse_prediction(resp, prompt_config))
            y_prob.append(p)
    elapsed = time.time() - t0
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred); y_prob = np.asarray(y_prob)
    metrics = compute_metrics(y_true, y_pred, y_prob, num_classes)
    boot = bootstrap_metrics(y_true, y_pred, y_prob, num_classes, n_iter=1000)
    return dict(metrics=metrics, bootstrap=boot, n_test=int(len(y_true)), time_total=elapsed)

In [ ]:
all_results = {}
t_eval = time.time()
for name in DATASETS_TO_USE:
    print(f"\n=== {name} ===")
    res = evaluate_dataset(name, loaded[name], model_lora, tokenizer, device)
    all_results[name] = res
    print("  Метрики:")
    for k, v in res["metrics"].items():
        if isinstance(v, float) and not np.isnan(v):
            print(f"    {k}: {v:.4f}")
        else:
            print(f"    {k}: {v}")
    print("  Bootstrap (mean±std):")
    for k, v in res["bootstrap"].items():
        print(f"    {k}: {v}")
print(f"\nВся оценка заняла {(time.time()-t_eval)/60:.1f} мин")

# 5. Сводная таблица и сохранение

In [ ]:
rows = []
for name, res in all_results.items():
    rows.append(dict(
        dataset=name,
        **{k: res["metrics"][k] for k in ["ROC-AUC", "F1", "Accuracy", "Precision", "Recall"]},
        n_test=res["n_test"],
    ))
summary = pd.DataFrame(rows).sort_values("dataset").reset_index(drop=True)
summary

In [ ]:
serializable = {
    name: dict(
        metrics={k: (None if isinstance(v, float) and np.isnan(v) else float(v))
                 for k, v in res["metrics"].items()},
        bootstrap=res["bootstrap"],
        n_test=res["n_test"],
        time_total=res["time_total"],
    )
    for name, res in all_results.items()
}
serializable["_meta"] = dict(
    model=model_name,
    datasets=DATASETS_TO_USE,
    per_dataset_cap=PER_DATASET_CAP,
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    grad_accum=GRAD_ACCUM,
    max_seq_len=MAX_SEQ_LEN,
    learning_rate=LEARNING_RATE,
    per_dataset_train_counts=per_dataset_counts,
    train_seconds=train_time,
)
with open(RESULTS_FILE, "w") as f:
    json.dump(serializable, f, indent=2, ensure_ascii=False)
print(f"Результаты: {RESULTS_FILE}")
print(f"Лог: {LOG_FILE}")